In [1]:
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [2]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("⚠️  OPENAI_API_KEY not found in .env file")
else:
    print("✅ API key loaded successfully")

✅ API key loaded successfully


In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict


class LLMProcessingState(TypedDict):
    user_input: str
    llm_response: str
    processing_steps: list[str]

In [4]:
def llm_step(state: LLMProcessingState):
    # Initialize the LLM
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    # Create a message and call the LLM
    message = HumanMessage(content=state["user_input"])
    response = model.invoke([message])

    return {
        "llm_response": response.content,
        "processing_steps": state["processing_steps"] + ["Called LLM"],
    }


def format_output(state: LLMProcessingState):
    formatted = (
        f"User asked: {state['user_input']}\n\nLLM replied: {state['llm_response']}"
    )
    return {
        "llm_response": formatted,
        "processing_steps": state["processing_steps"] + ["Formatted output"],
    }

In [5]:
# Build graph
graph = StateGraph(LLMProcessingState)
graph.add_node("llm", llm_step)
graph.add_node("format", format_output)

graph.set_entry_point("llm")
graph.add_edge("llm", "format")
graph.add_edge("format", END)

# Compile
app = graph.compile()

# Test it
result = app.invoke(
    {
        "user_input": "What is LangGraph and why is it useful?",
        "llm_response": "",
        "processing_steps": [],
    }
)

print(result["llm_response"])
print("\nSteps taken:", result["processing_steps"])

User asked: What is LangGraph and why is it useful?

LLM replied: LangGraph is a framework designed to facilitate the development of applications that leverage large language models (LLMs) in conjunction with graph-based data structures. It aims to simplify the integration of LLMs with knowledge graphs or other types of graph data, enabling developers to create more sophisticated and context-aware applications.

### Key Features and Benefits of LangGraph:

1. **Integration of LLMs and Graphs**: LangGraph allows developers to seamlessly combine the generative capabilities of LLMs with the structured information provided by graphs, enabling applications to draw on both free-form text and structured data.

2. **Enhanced Contextual Understanding**: By using graph structures, LangGraph can provide LLMs with additional context that may not be readily available in plain text. This can lead to more accurate and context-aware responses.

3. **Improved Knowledge Retrieval**: With the ability to 